# Homework 1: scraping and crawling

Information Retrieval, Innopolis University.

You write a small crawler and the pieces around it: a polite fetcher with a
cache, a page parser, a bit of text processing, the crawler itself, and a
scraper for a shop that hides its data behind a JSON API.

Everything runs against a practice website that starts on your own machine, so
nothing will depend on a real site, on your internet connection or on
somebody's rate limit.

## Rules

- Rename this file to your university email plus `.ipynb`, for example
  `i.ivanov@innopolis.university.ipynb`. A wrong file name means 0 points.
- Submit the notebook file only. Keep `ir_sandbox.py` next to it while you work,
  but do not submit it and do not change it.
- The grader imports your notebook and calls your functions. Only imports,
  function definitions, class definitions and simple constant assignments
  survive that import. Code you run at the top level of a cell (calls, prints,
  experiments) is ignored, so never define anything important there.
- Keep the names and the signatures exactly as given. A renamed function is a
  missing function.
- Allowed libraries: the Python standard library, `requests` and
  `beautifulsoup4`. No `lxml`, no `html5lib`, no `nltk`, no Selenium, no
  `aiohttp`, no `!pip install`. Use `BeautifulSoup(html, "html.parser")`.
- Python 3.10 or newer.

## Points

| Task | What | Points |
| --- | --- | --- |
| 1 | `PoliteFetcher`: cache, user agent, delays, timeouts, streamed download | 20 |
| 2 | `normalize_url` and `parse_page` | 20 |
| 3 | sentences, terms, ranking, two metrics | 15 |
| 4 | `Crawler` | 30 |
| 5 | shop behind a JSON API | 15 |

Partial credit is provided, so do try not to leave empty cells.
## Interview

After the deadline every student meets a TA for a few minutes and answers a
question or two about this code: why a function does what it does, what it
returns for an input you have not seen, what happens if the site answers
differently. The point is that you can explain and change your own work. You
may use any tool you like while writing it, including AI assistants, as long as
you understand every line you hand in.

## Self-check

`ir_sandbox.check(n, globals())` runs the same tests the grader runs, and the
final cell of this notebook grades the saved file the same way. We run them on
another copy of the site: same kind of pages, same rules, other addresses, other
texts and other products. So write code that follows the task, not code that
matches the pages you see here. Anything that hardcodes a path, a title or a
product of the practice site passes the self-check and fails the real run.


## The practice website

Run the next cell. It starts two small websites on your own machine:

- the museum site, the one you crawl
- a partner archive on another port, which has its own `robots.txt` with a
  `Crawl-delay`

It also prints an address where nothing is listening, which is handy for
testing connection errors, and a few test addresses for slow answers, error
codes and large downloads.

You can open the museum address in a browser and click around. Reading the page
source is part of the work. `SITE.requests()` gives you every request the site
has received, which is the easiest way to see what your code is really doing.


In [1]:
import ir_sandbox

SITE = ir_sandbox.start()


Practice site is running (ir_sandbox 2026.1).
  museum site     http://127.0.0.1:8765/
  partner site    http://127.0.0.1:8766/
  nothing here    http://127.0.0.1:52709/  (use it to test connection errors)
Useful addresses for Task 1:
  slow page       http://127.0.0.1:8765/_test/slow?seconds=3
  any status      http://127.0.0.1:8765/_test/status/404  (try 500, 503 and so on)
  your headers    http://127.0.0.1:8765/_test/headers
  downloads       http://127.0.0.1:8765/downloads/small.bin  big.bin  broken.bin  tampered.bin
Open the museum site in your browser to look around.
SITE.requests() shows every request the site received.


The next cell is given. Put your email in `USER_AGENT`, that is the only change you need here.

In [3]:
# Put your own email in USER_AGENT. Everything else in this cell is given.
import hashlib
import json
import math
import os
import re
import time
from collections import Counter, deque
from typing import Any, Dict, Iterator, List, Optional, Set, Tuple
from urllib.parse import urljoin, urlsplit, urlunsplit
from urllib.robotparser import RobotFileParser

import requests
from bs4 import BeautifulSoup

USER_AGENT = "IRHomeworkBot/1.0 (+ma.chegodaeva@innopolis.university)"

# Largest piece of a download that may sit in memory at once.
CHUNK_SIZE = 256 * 1024

# The crawler never follows links whose path ends with one of these.
SKIP_EXTENSIONS = (".pdf", ".png", ".jpg", ".jpeg", ".gif", ".svg", ".zip", ".mp3", ".mp4")

# Words that carry no meaning for ranking (Task 3).
STOPWORDS = {
    "a", "an", "and", "are", "as", "at", "be", "by", "can", "did", "do", "does",
    "for", "from", "has", "have", "how", "i", "in", "is", "it", "its", "many",
    "much", "my", "of", "on", "or", "that", "the", "their", "this", "to", "was",
    "were", "what", "when", "where", "which", "who", "why", "will", "with",
    "you", "your",
}

# A single dot after one of these words does not end a sentence (Task 3).
ABBREVIATIONS = {"dr", "mr", "mrs", "ms", "prof", "st", "mt", "vs", "no", "fig", "approx", "e.g", "i.e"}

## Task 1: `PoliteFetcher` (20 points)

A fetcher that is safe to point at a real website: it identifies itself, waits
between requests, times out, keeps a copy of what it downloads, and never
crashes on a broken server.

```python
class PoliteFetcher:
    def __init__(self, cache_dir="cache", user_agent=USER_AGENT, delay=0.0, timeout=5.0)
    def cache_paths(self, url) -> tuple[str, str]
    def set_host_delay(self, netloc, seconds) -> None
    def fetch(self, url) -> dict
    def download(self, url, path) -> str | None
```

`fetch(url)` returns a dict with exactly these keys:

- `url`: the URL you were asked to fetch, unchanged
- `final_url`: the URL after redirects, or `url` when the request failed
- `status`: the HTTP status code, or `None` when there was no answer at all
  (timeout, connection error, too many redirects)
- `content_type`: the part of the `Content-Type` header before the first
  semicolon, lowercase and stripped, `""` when there was no answer
- `body`: the response bytes when the status is 200, otherwise `None`
- `from_cache`: `True` when the answer came from disk

Rules:

* Only responses with status 200 should be saved in the cache. Store the body in one file and the other response fields in a small JSON file next to it. `cache_paths(url)` should return these two paths, and both files must be inside `cache_dir`. URLs that differ in any way, including the query string, must use different files.

* If a new `PoliteFetcher` is created with the same `cache_dir`, it should be able to return a cached response from disk without making a network request. It should return the same status, content type, final URL, and body as the original response.

* Every request should use the user agent given to the constructor. Store it in `self.user_agent`, since the crawler in Task 4 reads it from there.

* Wait at least `delay` seconds between requests to the same host. `set_host_delay(netloc, seconds)` should increase the delay for one specific host. The `netloc` looks like `127.0.0.1:8765`. Delays are tracked separately for each host, and cache hits should never cause a wait.

* Pass `timeout` to `requests`. Also, `fetch` must never raise an exception.

* `download(url, path)` should stream the response body to disk using chunks no larger than `CHUNK_SIZE` bytes and return the SHA-256 hex digest of the downloaded data. The entire file must never be kept in memory. The tests download 32 MiB and use `tracemalloc` to check memory usage.

* `download(url, path)` should return `None` and leave no file behind when the response status is not 200, when the number of received bytes does not match `Content-Length`, or when the server sends an `X-Checksum-SHA256` header and its value does not match the SHA-256 digest of the data received.

* Write the downloaded data to a temporary file first. Rename it to `path` only after the download finishes successfully. This makes sure that a failed download cannot leave a partial file at `path`.



In [5]:
import tempfile


class PoliteFetcher:
    """Downloads pages politely and keeps successful responses in a disk cache."""

    def __init__(self, cache_dir: str = "cache", user_agent: str = USER_AGENT,
                 delay: float = 0.0, timeout: float = 5.0) -> None:
        self.cache_dir = os.fspath(cache_dir)
        self.user_agent = user_agent
        self.delay = max(0.0, delay)
        self.timeout = timeout
        self.host_delays = {}
        self.last_request = {}
        os.makedirs(self.cache_dir, exist_ok=True)

    def cache_paths(self, url: str) -> Tuple[str, str]:
        """Hash the complete URL, including its query string."""
        key = hashlib.sha256(url.encode("utf-8")).hexdigest()
        return (os.path.join(self.cache_dir, key + ".body"),
                os.path.join(self.cache_dir, key + ".json"))

    def set_host_delay(self, netloc: str, seconds: float) -> None:
        host = netloc.lower()
        self.host_delays[host] = max(self.delay, self.host_delays.get(host, 0.0), seconds)

    def _request(self, url: str, stream: bool = False):
        # Follow redirects ourselves so that their requests also respect delays.
        headers = {"User-Agent": self.user_agent}
        if stream:
            headers["Accept-Encoding"] = "identity"
        for _ in range(31):
            host = urlsplit(url).netloc.lower()
            delay = self.host_delays.get(host, self.delay)
            if host in self.last_request:
                remaining = delay - (time.monotonic() - self.last_request[host])
                if remaining > 0:
                    time.sleep(remaining)
            self.last_request[host] = time.monotonic()
            response = requests.get(url, headers=headers, timeout=self.timeout,
                                    allow_redirects=False, stream=stream)
            if response.status_code not in (301, 302, 303, 307, 308) or not response.headers.get("Location"):
                return response
            url = urljoin(response.url, response.headers["Location"])
            response.close()
        raise requests.TooManyRedirects("Too many redirects")

    def _save_cache(self, result: Dict[str, Any]) -> None:
        body_path, meta_path = self.cache_paths(result["url"])
        metadata = {key: value for key, value in result.items() if key != "body"}
        # Each replacement is atomic; an incomplete cache entry is ignored on read.
        for destination, data in ((body_path, result["body"]),
                                  (meta_path, json.dumps(metadata).encode("utf-8"))):
            temporary = None
            try:
                with tempfile.NamedTemporaryFile(dir=self.cache_dir, delete=False) as handle:
                    temporary = handle.name
                    handle.write(data)
                os.replace(temporary, destination)
            finally:
                if temporary is not None and os.path.exists(temporary):
                    os.remove(temporary)

    def fetch(self, url: str) -> Dict[str, Any]:
        """Return url, final_url, status, content_type, body and from_cache."""
        result = {"url": url, "final_url": url, "status": None,
                  "content_type": "", "body": None, "from_cache": False}
        try:
            body_path, meta_path = self.cache_paths(url)
            try:
                with open(meta_path, encoding="utf-8") as handle:
                    metadata = json.load(handle)
                if metadata["status"] == 200 and metadata["url"] == url:
                    with open(body_path, "rb") as handle:
                        body = handle.read()
                    return {"url": url, "final_url": metadata["final_url"], "status": 200,
                            "content_type": metadata["content_type"], "body": body, "from_cache": True}
            except (OSError, ValueError, KeyError, TypeError):
                pass  # A missing or damaged cache entry can be downloaded again.

            with self._request(url) as response:
                result.update(final_url=response.url, status=response.status_code,
                              content_type=response.headers.get("Content-Type", "").split(";", 1)[0].strip().lower(),
                              body=response.content if response.status_code == 200 else None)
            if result["status"] == 200:
                try:
                    self._save_cache(result)
                except OSError:
                    pass  # A cache write failure does not invalidate the response.
        except Exception:
            # The task requires fetch to return a record even for request failures.
            pass
        return result

    def download(self, url: str, path: str) -> Optional[str]:
        """Stream to a temporary file; publish only after checking length and hash."""
        temporary = None
        try:
            with self._request(url, stream=True) as response:
                if response.status_code != 200:
                    return None
                digest = hashlib.sha256()
                received = 0
                folder = os.path.dirname(os.path.abspath(path))
                with tempfile.NamedTemporaryFile(dir=folder, delete=False) as handle:
                    temporary = handle.name
                    for chunk in response.iter_content(chunk_size=CHUNK_SIZE):
                        if chunk:
                            handle.write(chunk)
                            digest.update(chunk)
                            received += len(chunk)
                length = response.headers.get("Content-Length")
                checksum = response.headers.get("X-Checksum-SHA256")
                if length is not None and received != int(length):
                    return None
                if checksum is not None and digest.hexdigest() != checksum.strip().lower():
                    return None
            os.replace(temporary, path)
            temporary = None
            return digest.hexdigest()
        except Exception:
            return None
        finally:
            if temporary is not None:
                try:
                    os.remove(temporary)
                except OSError:
                    pass

In [6]:
# Try your fetcher. This cell is not graded.
fetcher = PoliteFetcher(cache_dir="cache_demo", user_agent=USER_AGENT, delay=0.2)
first = fetcher.fetch(SITE.url("/about/"))
print(first["status"], first["content_type"], len(first["body"] or b""), first["from_cache"])
print("second time from cache:", fetcher.fetch(SITE.url("/about/"))["from_cache"])
print("missing page:", fetcher.fetch(SITE.url("/events/2019-gala"))["status"])
print("nothing listening:", fetcher.fetch(SITE.closed + "/")["status"])
print(SITE.requests()[-3:])


200 text/html 1649 True
second time from cache: True
missing page: 404
nothing listening: None
[{'seq': 1, 'time': 174199.2357089, 'host': 'main', 'method': 'GET', 'path': '/events/2019-gala', 'user_agent': 'IRHomeworkBot/1.0 (+ma.chegodaeva@innopolis.university)', 'status': 404}]


In [7]:
ir_sandbox.check(1, globals())

Task 1  Polite fetcher   20/20 points
  ok       2/2   fetch returns an HTML page with all fields
  ok       2/2   every request sends the User-Agent
  ok       2/2   binary files come back unchanged
  ok       3/3   disk cache: one entry per URL, reused by a new fetcher
  ok       2/2   error pages are reported and never cached
  ok       1/1   redirects are followed and cached under the requested URL
  ok       2/2   timeouts and connection errors give status None
  ok       2/2   delay between requests to the same host
  ok       4/4   download streams big files safely


## Task 2: URLs and page parsing (20 points)

```python
def normalize_url(url, base=None) -> str | None
def parse_page(html, page_url) -> dict
```

`normalize_url` takes whatever is in an `href` and turns it into one canonical form so the crawler can compare URLs.

* Strip spaces around the URL. When `base` is given, resolve the URL against it with `urljoin`.

* Convert the scheme and host to lowercase. Remove `user:password@`. Remove the port when it is the default one for the scheme, which is 80 for http and 443 for https. Keep any other port.

* Return `None` for anything that is not http or https, and for URLs that have no host.

* Remove the fragment.

* When the path is empty, use `/`. Resolve dot segments according to RFC 3986 section 5.2.4. For example, `/a/b/../c` becomes `/a/c`.

* Remove query parameters whose name starts with `utm_` in any case. Remove empty parameters. Keep all other parameters in their original order. When nothing is left, remove the query string completely.

* Never raise an exception. Return `None` when the URL cannot be parsed.

`parse_page(html, page_url)` should return a dict with the keys `title`, `links`, `images`, `paragraphs`, `text`, `canonical` and `meta_robots`.

* Parse the HTML with `BeautifulSoup(html, "html.parser")`.

* Before reading anything, remove every `script`, `style`, `noscript` and `template` element, as well as every element with a `hidden` attribute. One removed element may be inside another removed element, so handle nested removed elements correctly.

* The base URL is the `href` of the first `base` tag resolved against `page_url`. When there is no `base` tag, use `page_url`. Resolve every URL on the page against this base URL, including an empty `href`.

* `title` is the text inside `title` with whitespace collapsed. Use `""` when there is no title.

* `links` contains one dict for each `a` element that has an `href`, in document order. Each dict has `text`, `url` and `nofollow`. `text` is the anchor text with whitespace collapsed. `url` is the normalized URL. `nofollow` is `True` when the `rel` attribute contains `nofollow` in any case. Skip links when `normalize_url` returns `None`.

* `images` contains the normalized URLs from `img` elements. Use `data-src` when it exists and is not blank. Otherwise use `src`. Skip blank values and duplicates, but keep the original document order.

* `paragraphs` contains the text of every `p` element inside `main`. When there is no `main`, use `body` instead. Collapse whitespace and remove empty paragraphs.

* `text` contains all visible text from `body`, with all whitespace collapsed into single spaces.

* `canonical` is the normalized `href` from `link rel="canonical"`. The `rel` value can use any letter case. Use `None` when there is no canonical link.

* `meta_robots` is a set of lowercase directives from `meta name="robots"`, split by commas. Use an empty set when the tag is missing.

In [8]:
from urllib.parse import unquote_plus


def _remove_dot_segments(path: str) -> str:
    """RFC 3986 section 5.2.4, preserving repeated and trailing slashes."""
    output = ""
    while path:
        if path.startswith("../"):
            path = path[3:]
        elif path.startswith("./"):
            path = path[2:]
        elif path.startswith("/./") or path == "/.":
            path = "/" + path[3:]
        elif path.startswith("/../") or path == "/..":
            path = "/" + path[4:]
            output = output.rsplit("/", 1)[0]
        elif path in (".", ".."):
            path = ""
        else:
            end = path.find("/", 1 if path.startswith("/") else 0)
            if end == -1:
                output += path
                path = ""
            else:
                output += path[:end]
                path = path[end:]
    return output


def normalize_url(url: str, base: Optional[str] = None) -> Optional[str]:
    """Resolve and canonicalize a URL, or return None if it is not http(s)."""
    try:
        url = url.strip()
        if base is not None:
            url = urljoin(base, url)
        parts = urlsplit(url)
        scheme = parts.scheme.lower()
        host = parts.hostname
        if scheme not in ("http", "https") or not host or any(c.isspace() for c in host):
            return None
        host = host.lower()
        port = parts.port  # Also validates malformed and out-of-range ports.
        if ":" in host:
            host = "[" + host + "]"
        if port is not None and (scheme, port) not in (("http", 80), ("https", 443)):
            host += ":" + str(port)
        parameters = []
        for parameter in parts.query.split("&"):
            name = unquote_plus(parameter.split("=", 1)[0]).lower()
            if parameter and not name.startswith("utm_"):
                parameters.append(parameter)
        path = _remove_dot_segments(parts.path) or "/"
        return urlunsplit((scheme, host, path, "&".join(parameters), ""))
    except Exception:
        return None


def _clean_text(element, separator: str = " ") -> str:
    return " ".join(element.get_text(separator).split()) if element is not None else ""


def _rel_values(element) -> Set[str]:
    values = element.get("rel", [])
    if isinstance(values, str):
        values = values.split()
    return {value.lower() for value in values}


def parse_page(html: bytes, page_url: str) -> Dict[str, Any]:
    """Return title, links, images, paragraphs, text, canonical and meta_robots."""
    soup = BeautifulSoup(html, "html.parser")
    # Remove children before parents, since decompose also destroys descendants.
    for element in reversed(soup.select("script, style, noscript, template, [hidden]")):
        element.decompose()
    base_tag = soup.find("base", href=True)
    base = urljoin(page_url, base_tag["href"]) if base_tag else page_url
    links = []
    for anchor in soup.find_all("a", href=True):
        url = normalize_url(anchor["href"], base)
        if url is not None:
            links.append({"text": _clean_text(anchor), "url": url,
                          "nofollow": "nofollow" in _rel_values(anchor)})
    images = []
    seen_images = set()
    for element in soup.find_all("img"):
        source = element.get("data-src", "").strip() or element.get("src", "").strip()
        url = normalize_url(source, base) if source else None
        if url is not None and url not in seen_images:
            seen_images.add(url)
            images.append(url)
    body = soup.body if soup.body is not None else soup
    main = soup.find("main")
    paragraph_root = main if main is not None else body
    # Inline tags must not introduce spaces before punctuation in a paragraph.
    paragraphs = [_clean_text(p, separator="") for p in paragraph_root.find_all("p")]
    canonical = None
    for element in soup.find_all("link", href=True):
        if "canonical" in _rel_values(element):
            canonical = normalize_url(element["href"], base)
            break
    meta_robots = set()
    for element in soup.find_all("meta"):
        if element.get("name", "").lower() == "robots":
            meta_robots.update(directive.strip().lower()
                               for directive in element.get("content", "").split(",")
                               if directive.strip())
    return {"title": _clean_text(soup.title), "links": links, "images": images,
            "paragraphs": [p for p in paragraphs if p], "text": _clean_text(body),
            "canonical": canonical, "meta_robots": meta_robots}

In [9]:
# Try your parser. This cell is not graded.
page = PoliteFetcher(cache_dir="cache_demo", user_agent=USER_AGENT).fetch(SITE.url("/halls/rivers"))
parsed = parse_page(page["body"], SITE.url("/halls/rivers"))
print(parsed["title"], "|", parsed["canonical"], "|", sorted(parsed["meta_robots"]))
for link in parsed["links"][6:9]:
    print(link)
print(parsed["images"])
print(parsed["paragraphs"][0])


Volga Rivers hall | http://127.0.0.1:8765/halls/rivers | []
{'text': 'Russian sturgeon', 'url': 'http://127.0.0.1:8765/collections/rivers/sturgeon', 'nofollow': False}
{'text': 'a map of the delta', 'url': 'http://127.0.0.1:8765/collections/rivers/delta-map', 'nofollow': True}
{'text': 'a paddle steamer model', 'url': 'http://127.0.0.1:8765/exhibits/3', 'nofollow': False}
['http://127.0.0.1:8765/files/logo.png', 'http://127.0.0.1:8765/collections/rivers/img/sturgeon.png']
Our rivers hall follows the Volga from its source to the Caspian Sea. The sturgeon tank holds 12 fish, and the oldest one is 31 years old.


In [10]:
ir_sandbox.check(2, globals())

Task 2  URL normalization and page parsing   20/20 points
  ok       6/6   normalize_url (6 groups of cases)
  ok       2/2   title, canonical and meta_robots
  ok       2/2   links: URLs in document order
  ok       1/1   links: anchor text
  ok       1/1   links: nofollow flags
  ok       2/2   images
  ok       3/3   paragraphs
  ok       2/2   visible text
  ok       1/1   nothing from scripts, styles, templates or hidden elements


## Task 3: sentences, terms and ranking (15 points)

```python
def split_sentences(text) -> list[str]
def tokenize(text) -> list[str]
def top_terms(texts, n=10) -> list[tuple[str, int]]
def rank_sentences(question, sentences) -> list[int]
def precision_at_k(ranked, relevant, k) -> float
def reciprocal_rank(ranked, relevant, k) -> float
def best_answer(question, pages) -> tuple[str, str] | None
```
`split_sentences` should split the text into sentences.

* A sentence ends at `.`, `!` or `?`. These characters can appear more than once.

* They can be followed by closing quotes or brackets, then whitespace, and then an uppercase letter, a digit, or an opening quote or bracket.

* A single `.` does not end a sentence when the word before it is in `ABBREVIATIONS` or when that word has only one letter. Compare the word in lowercase after removing any opening brackets and quotes from the beginning.

* Strip each sentence, remove empty ones, and do not change anything else inside the sentences.

`tokenize` should lowercase the text and return all runs of letters and digits using `re.findall(r"[^\W_]+", text.lower())`. Letters outside the Latin alphabet should also count as letters.

`top_terms` should count all tokens from all texts together.

* Remove tokens from `STOPWORDS`.
* Remove tokens that contain only digits.
* Sort first by count from highest to lowest, then alphabetically by the term.
* Return the first `n` pairs.

`rank_sentences` should rank sentences based on the question.

* The query terms are `tokenize(question)` with anything in `STOPWORDS` removed.

* A sentence gets a score by adding the score of every query term that appears in it. Each term counts only once per sentence.

* The score for each term is `log((1 + N) / (1 + df)) + 1`, where `N` is the total number of sentences and `df` is the number of sentences containing that term.

* Remove sentences with a score of 0.

* Return the indices of the remaining sentences, sorted by highest score first. When scores are tied, sort by fewer tokens first, then by the smaller index.

`precision_at_k` is the number of the first `k` items in `ranked` that are also in `relevant`, divided by `k`.

`reciprocal_rank` is `1` divided by the position of the first relevant item among the first `k` items in `ranked`, counting positions from 1. Return `0.0` when there is no relevant item.

`best_answer` takes `pages`, which is a list of dicts containing `url` and `paragraphs`.

* Split every paragraph into sentences.
* Rank all sentences together.
* Return the best matching sentence as `(sentence, url of its page)`.
* Return `None` when no sentence matches.

In [11]:
def split_sentences(text: str) -> List[str]:
    sentences = []
    start = 0
    # Keep punctuation and closing quotes/brackets in the preceding sentence.
    boundary = r"([.!?]+)[\"'\u201d\u2019\u00bb)\]}]*\s+"
    for match in re.finditer(boundary, text):
        if match.end() == len(text):
            continue
        following = text[match.end()]
        if not (following.isupper() or following.isdigit() or following in "\"'\u201c\u2018\u00ab([{"):
            continue
        if match.group(1) == ".":
            word_match = re.search(r"([^\W\d_]+(?:\.[^\W\d_]+)*)$", text[:match.start()])
            word = word_match.group(1).lower() if word_match else ""
            if word in ABBREVIATIONS or len(word) == 1:
                continue
        sentence = text[start:match.end()].strip()
        if sentence:
            sentences.append(sentence)
        start = match.end()
    remainder = text[start:].strip()
    if remainder:
        sentences.append(remainder)
    return sentences


def tokenize(text: str) -> List[str]:
    return re.findall(r"[^\W_]+", text.lower())


def top_terms(texts: List[str], n: int = 10) -> List[Tuple[str, int]]:
    counts = Counter(term for text in texts for term in tokenize(text)
                     if term not in STOPWORDS and not term.isdigit())
    return sorted(counts.items(), key=lambda item: (-item[1], item[0]))[:max(0, n)]


def rank_sentences(question: str, sentences: List[str]) -> List[int]:
    """Indices of the matching sentences, best first."""
    query = sorted(set(tokenize(question)) - STOPWORDS)
    tokens = [tokenize(sentence) for sentence in sentences]
    terms = [set(sentence_tokens) for sentence_tokens in tokens]
    frequencies = Counter(term for sentence_terms in terms for term in sentence_terms)
    weights = {term: math.log((1 + len(sentences)) / (1 + frequencies[term])) + 1
               for term in query}
    scores = [sum(weights[term] for term in query if term in sentence_terms)
              for sentence_terms in terms]
    matches = [index for index, score in enumerate(scores) if score > 0]
    return sorted(matches, key=lambda index: (-scores[index], len(tokens[index]), index))


def precision_at_k(ranked: List[int], relevant: Set[int], k: int) -> float:
    if k <= 0:
        return 0.0
    return sum(item in relevant for item in ranked[:k]) / k


def reciprocal_rank(ranked: List[int], relevant: Set[int], k: int) -> float:
    if k <= 0:
        return 0.0
    for position, item in enumerate(ranked[:k], start=1):
        if item in relevant:
            return 1.0 / position
    return 0.0


def best_answer(question: str, pages: List[Dict[str, Any]]) -> Optional[Tuple[str, str]]:
    """Best sentence over all pages, as (sentence, page url)."""
    candidates = [(sentence, page["url"]) for page in pages
                  for paragraph in page["paragraphs"] for sentence in split_sentences(paragraph)]
    ranked = rank_sentences(question, [sentence for sentence, url in candidates])
    return candidates[ranked[0]] if ranked else None

In [12]:
# Try your text functions. This cell is not graded.
print(split_sentences("Dr. Petrova found it in 1978. The skeleton is 3.2 m tall! Where? Near Kazan."))
print(tokenize("The Mammoth's tusk: 2.4 m!"))
print(top_terms(["rivers and fish", "fish, fish and lakes"], 3))
print(rank_sentences("how tall is the skeleton", ["The skeleton is 3.2 m tall.", "Tickets cost 150 rubles."]))
print(precision_at_k([3, 1, 2], {1, 2}, 2), reciprocal_rank([4, 2, 7], {7}, 3))


['Dr. Petrova found it in 1978.', 'The skeleton is 3.2 m tall!', 'Where?', 'Near Kazan.']
['the', 'mammoth', 's', 'tusk', '2', '4', 'm']
[('fish', 3), ('lakes', 1), ('rivers', 1)]
[0]
0.5 0.3333333333333333


In [14]:
ir_sandbox.check(3, globals())

Task 3  Sentences, terms and ranking   15/15 points
  ok       4/4   split_sentences (4 groups of cases)
  ok       2/2   tokenize
  ok       2/2   top_terms
  ok       4/4   rank_sentences (order 2, ties 1, empty cases 1)
  ok       1/1   precision_at_k and reciprocal_rank
  ok       2/2   best_answer on the museum pages


## Task 4: the crawler (30 points)

```python
class Crawler:
    def __init__(self, start_url, fetcher, max_pages=20, max_depth=2, allowed_hosts=None)
    def crawl(self)          # generator
    self.errors              # list of (url, status)
    self.blocked             # list of URLs robots.txt kept you away from
```
Your crawler may only use `fetcher.fetch(url)`, `fetcher.set_host_delay(netloc, seconds)` and `fetcher.user_agent` from the fetcher. You may also use your own `normalize_url` and `parse_page`. We test this task with our own fetcher and parser, so bugs in Task 1 or Task 2 will not be counted against you again here.

* Normalize `start_url`. When `allowed_hosts` is `None`, only the host of the start URL is allowed. A host is written as a netloc such as `127.0.0.1:8765`.

* Crawl in breadth-first order using a FIFO queue. The start page has depth 0.

* Before making the first request to a host, fetch that host's `/robots.txt` through the fetcher. Read it with `urllib.robotparser.RobotFileParser().parse(text.splitlines())`. Do this only once for each host. If `robots.txt` is missing or does not return status 200, allow everything on that host. When it defines a `Crawl-delay` for your user agent, pass that delay to `fetcher.set_host_delay`.

* When `robots.txt` forbids a URL, add it to `self.blocked` and never request it.

* Each page request counts as one attempt. Requests for `robots.txt` do not count. Stop when you have made `max_pages` page attempts or when the queue is empty.

* When a page returns a status other than 200, add `(url, status)` to `self.errors` and do not yield anything for that page.

* When the final URL after redirects is different from the requested URL, skip the page when the final URL is outside `allowed_hosts` or has already been seen. Otherwise, mark the final URL as seen and use the final URL in the returned record.

* Only parse pages whose content type is `text/html`.

* Yield `{"url": final url, "depth": depth, "title": title, "links": [link urls in document order]}` unless the page has `noindex` in its meta robots.

* Do not follow any links when the page has `nofollow` in its meta robots, or when its depth is already `max_depth`.

* Skip links when they have `rel="nofollow"`, have already been seen, are outside `allowed_hosts`, or have a path ending with one of the extensions in `SKIP_EXTENSIONS`.

* `crawl()` must be a generator. It should fetch the next page only when the caller asks for the next record. For example, `itertools.islice(crawler.crawl(), 2)` should make only two page requests, not twenty.

The practice site is designed to test these cases: a redirect, a redirect loop, a 404, a 500, a page with `noindex`, a page with `nofollow`, a `base` tag, an endless calendar, a PDF, a JSON endpoint and a partner host with a crawl delay.


In [15]:
class Crawler:
    """Breadth-first crawler that yields one record per indexable HTML page."""

    def __init__(self, start_url: str, fetcher: "PoliteFetcher", max_pages: int = 20,
                 max_depth: int = 2, allowed_hosts: Optional[Set[str]] = None) -> None:
        self.errors: List[Tuple[str, Optional[int]]] = []
        self.blocked: List[str] = []
        self.start_url = normalize_url(start_url)
        self.fetcher = fetcher
        self.max_pages = max_pages
        self.max_depth = max_depth
        if allowed_hosts is None:
            self.allowed_hosts = {urlsplit(self.start_url).netloc} if self.start_url else set()
        else:
            self.allowed_hosts = {host.lower() for host in allowed_hosts}
        self.robots = {}

    def can_fetch(self, url: str) -> bool:
        """Load robots.txt once per host, before requesting any of its pages."""
        parts = urlsplit(url)
        host = parts.netloc
        if host not in self.robots:
            robots_url = urlunsplit((parts.scheme, host, "/robots.txt", "", ""))
            response = self.fetcher.fetch(robots_url)
            parser = None
            if response["status"] == 200:
                parser = RobotFileParser()
                parser.parse((response["body"] or b"").decode("utf-8", errors="replace").splitlines())
                delay = parser.crawl_delay(self.fetcher.user_agent)
                if delay is not None:
                    self.fetcher.set_host_delay(host, delay)
            self.robots[host] = parser
        parser = self.robots[host]
        return parser is None or parser.can_fetch(self.fetcher.user_agent, url)

    def crawl(self) -> Iterator[Dict[str, Any]]:
        """Yield {"url", "depth", "title", "links"} for every indexable page."""
        if self.start_url is None or self.max_depth < 0:
            return
        queue = deque([(self.start_url, 0)])
        queued = {self.start_url}
        seen = set()
        attempts = 0
        while queue and attempts < self.max_pages:
            url, depth = queue.popleft()
            if url in seen or urlsplit(url).netloc not in self.allowed_hosts:
                continue
            seen.add(url)
            if not self.can_fetch(url):
                self.blocked.append(url)
                continue
            attempts += 1
            response = self.fetcher.fetch(url)
            if response["status"] != 200:
                self.errors.append((url, response["status"]))
                continue
            final_url = normalize_url(response["final_url"])
            if final_url is None or urlsplit(final_url).netloc not in self.allowed_hosts:
                continue
            if final_url != url and final_url in seen:
                continue
            seen.add(final_url)
            if response["content_type"] != "text/html":
                continue
            parsed = parse_page(response["body"], final_url)
            if "noindex" not in parsed["meta_robots"]:
                yield {"url": final_url, "depth": depth, "title": parsed["title"],
                       "links": [link["url"] for link in parsed["links"]]}
            if depth >= self.max_depth or "nofollow" in parsed["meta_robots"]:
                continue
            for link in parsed["links"]:
                target = link["url"]
                parts = urlsplit(target)
                if (link["nofollow"] or target in seen or target in queued
                        or parts.netloc not in self.allowed_hosts
                        or parts.path.lower().endswith(SKIP_EXTENSIONS)):
                    continue
                queue.append((target, depth + 1))
                queued.add(target)

In [16]:
# Try your crawler. This cell is not graded.
fetcher = PoliteFetcher(cache_dir="cache_crawl", user_agent=USER_AGENT)
crawler = Crawler(SITE.url("/"), fetcher, max_pages=8, max_depth=2)
for record in crawler.crawl():
    print(record["depth"], record["url"], "|", record["title"], "|", len(record["links"]), "links")
print("blocked by robots.txt:", crawler.blocked)
print("errors:", crawler.errors)


0 http://127.0.0.1:8765/ | Innopolis Museum of Natural History | 16 links
1 http://127.0.0.1:8765/halls/ | Halls & galleries | 13 links
1 http://127.0.0.1:8765/events/ | Events | 13 links
1 http://127.0.0.1:8765/shop/ | Museum shop | 9 links
1 http://127.0.0.1:8765/about/ | About the museum | 15 links
1 http://127.0.0.1:8765/halls/rivers | Volga Rivers hall | 11 links
1 http://127.0.0.1:8765/search?q=mammoth | Search results for “mammoth” | 11 links
1 http://127.0.0.1:8765/halls/ice-age | Ice Age hall | 10 links
blocked by robots.txt: []
errors: []


In [17]:
ir_sandbox.check(4, globals())

Task 4  Crawler   30/30 points
  ok       6/6   yields exactly the right pages
  ok       4/4   breadth-first order and depths
  ok       3/3   records have url, depth, title and links
  ok       4/4   robots.txt is respected
  ok       3/3   requests only what it needs, each page once
  ok       2/2   errors and redirects
  ok       2/2   meta robots noindex and nofollow
  ok       2/2   max_pages and max_depth
  ok       2/2   crawl() is lazy
  ok       2/2   two hosts and Crawl-delay


## Task 5: the shop behind a JSON API (15 points)

The shop page shows nothing useful in its HTML: the products arrive later from
a JSON API. Open `SITE.url("/shop/")` in a browser, look at the page source,
and you will find the configuration the page itself uses.

```python
def parse_price(value) -> float | None
def normalize_product(raw) -> dict
def scrape_shop(shop_url, user_agent=USER_AGENT) -> list[dict]
def filter_products(products, tag=None, max_price=None, min_rating=None, in_stock_only=False) -> list[int]
```

`parse_price` should handle the values that the API actually sends.

* When the value is an `int` or `float`, return it as a float rounded to two decimal places. `True` does not count as a number.
* When the value is a string, keep only digits, commas and dots and remove everything else.
* When both a comma and a dot are present, treat the comma as a thousands separator.
* When there are only commas, treat a single comma followed by exactly two digits as a decimal comma. Otherwise, treat the commas as thousands separators.
* Return `None` when there is nothing sensible left after cleaning the value.

`normalize_product(raw)` should return:

* `id`: an `int`
* `name`: the title with whitespace collapsed
* `price`: the result of `parse_price` on the price field
* `rating`: a float out of 5. For example, `"4,5"` becomes 4.5 and `"9/10"` becomes 4.5. An empty or missing rating becomes `None`.
* `tags`: lowercase and stripped, with empty strings and duplicates removed. Keep the original order. When tags are given as a string, split them on commas.
* `in_stock`: `True` for `True`, for numbers greater than zero, and for the strings `yes`, `in stock` and `available` in any case. Otherwise return `False`.

`scrape_shop` should load the shop page, get the endpoint, header name and token from the configuration, and then walk through the API.

* Send the token in the header whose name is given in the configuration on every API request.
* The token changes every time the shop page is loaded, so use the token from the page you just loaded.
* Follow `next` until it becomes `null`. Resolve each `next` URL against the URL of the response where that `next` value was found.
* When the API returns `429` with a `Retry-After` header, wait for that many seconds and then repeat the same request.
* A product may appear on more than one page. Keep the first copy you see and preserve the order given by the API.
* Return `normalize_product` for every product.

`filter_products` should keep products that match every filter that is not `None`. The filters are a tag compared in lowercase, a maximum price, a minimum rating and an in-stock requirement.

* A product without a price does not pass a price filter.
* A product without a rating does not pass a rating filter.
* Sort first by rating from highest to lowest, with missing ratings last.
* Then sort by price from lowest to highest, with missing prices last.
* Then sort by `id`.
* Return the list of product ids.



In [18]:
def parse_price(value: Any) -> Optional[float]:
    if isinstance(value, bool):
        return None
    if isinstance(value, str):
        value = re.sub(r"[^0-9,.]", "", value)
        if "," in value and "." in value:
            value = value.replace(",", "")
        elif "," in value:
            if value.count(",") == 1 and len(value.rsplit(",", 1)[1]) == 2:
                value = value.replace(",", ".")
            else:
                value = value.replace(",", "")
    elif not isinstance(value, (int, float)):
        return None
    try:
        number = float(value)
        return round(number, 2) if math.isfinite(number) else None
    except (ValueError, OverflowError):
        return None


def parse_rating(value: Any) -> Optional[float]:
    """Convert a decimal rating or a fraction to the five-point scale."""
    if value is None or isinstance(value, bool):
        return None
    try:
        if isinstance(value, str):
            value = value.strip().replace(",", ".")
            if "/" in value:
                numerator, denominator = value.split("/", 1)
                value = 5 * float(numerator) / float(denominator)
        number = float(value)
        return number if math.isfinite(number) else None
    except (ValueError, TypeError, ZeroDivisionError, OverflowError):
        return None


def parse_stock(value: Any) -> bool:
    """Accept positive quantities and the specified availability strings."""
    if isinstance(value, (int, float)):
        return value > 0
    if isinstance(value, str):
        value = value.strip().lower()
        if value in {"yes", "in stock", "available"}:
            return True
        # API quantities can be strings too, as checked by the supplied grader.
        try:
            return float(value) > 0
        except ValueError:
            return False
    return False


def normalize_product(raw: Dict[str, Any]) -> Dict[str, Any]:
    raw_tags = raw.get("tags") or []
    if isinstance(raw_tags, str):
        raw_tags = raw_tags.split(",")
    tags = []
    for tag in raw_tags:
        tag = tag.strip().lower()
        if tag and tag not in tags:
            tags.append(tag)
    return {"id": int(raw["id"]), "name": " ".join(raw.get("title", "").split()),
            "price": parse_price(raw.get("price")), "rating": parse_rating(raw.get("rating")),
            "tags": tags, "in_stock": parse_stock(raw.get("stock"))}


def scrape_shop(shop_url: str, user_agent: str = USER_AGENT) -> List[Dict[str, Any]]:
    products = []
    seen_ids = set()
    with requests.Session() as session:
        session.headers.update({"User-Agent": user_agent})
        with session.get(shop_url, timeout=5) as response:
            response.raise_for_status()
            soup = BeautifulSoup(response.content, "html.parser")
            config = json.loads(soup.find("script", id="shop-config").string)
            next_url = urljoin(response.url, config["endpoint"])
        headers = {config["auth_header"]: config["token"]}
        while next_url is not None:
            with session.get(next_url, headers=headers, timeout=5) as response:
                if response.status_code == 429 and "Retry-After" in response.headers:
                    delay = max(0.0, float(response.headers["Retry-After"]))
                    response.close()
                    time.sleep(delay)
                    continue  # Retry the same URL with the same token.
                response.raise_for_status()
                data = response.json()
                for raw in data["items"]:
                    product_id = int(raw["id"])
                    if product_id not in seen_ids:
                        seen_ids.add(product_id)
                        products.append(normalize_product(raw))
                next_link = data["next"]
                next_url = urljoin(response.url, next_link) if next_link is not None else None
    return products


def filter_products(products: List[Dict[str, Any]], tag: Optional[str] = None,
                    max_price: Optional[float] = None, min_rating: Optional[float] = None,
                    in_stock_only: bool = False) -> List[int]:
    selected = []
    for product in products:
        if tag is not None and tag.strip().lower() not in product["tags"]:
            continue
        if max_price is not None and (product["price"] is None or product["price"] > max_price):
            continue
        if min_rating is not None and (product["rating"] is None or product["rating"] < min_rating):
            continue
        if in_stock_only and not product["in_stock"]:
            continue
        selected.append(product)
    selected.sort(key=lambda product: (
        -product["rating"] if product["rating"] is not None else math.inf,
        product["price"] if product["price"] is not None else math.inf,
        product["id"]))
    return [product["id"] for product in selected]

In [19]:
# Try your shop scraper. This cell is not graded.
products = scrape_shop(SITE.url("/shop/"), USER_AGENT)
print(len(products), "products")
for product in products[:3]:
    print(product)
print("cheap and in stock:", filter_products(products, max_price=1000, in_stock_only=True))


14 products
{'id': 101, 'name': 'Mammoth plush toy', 'price': 1299.0, 'rating': 4.8, 'tags': ['kids', 'toys'], 'in_stock': True}
{'id': 102, 'name': 'Volga river map', 'price': 450.0, 'rating': 4.1, 'tags': ['posters', 'maps'], 'in_stock': True}
{'id': 103, 'name': 'Fossil hunting kit', 'price': 2490.5, 'rating': 4.5, 'tags': ['kids', 'science'], 'in_stock': False}
cheap and in stock: [112, 105, 111, 102, 113, 104]


In [20]:
ir_sandbox.check(5, globals())

Task 5  Shop behind a JSON API   15/15 points
  ok       3/3   parse_price
  ok       3/3   normalize_product
  ok       5/5   scrape_shop collects the whole catalog
  ok       4/4   filter_products (4 queries)


## Before you submit

1. Restart the kernel and run the whole notebook from the top. The grader
   imports a fresh copy of your code, so anything that only works because of a
   variable you created by hand will fail there.
2. Save the notebook under your email name, for example
   `i.ivanov@innopolis.university.ipynb`.
3. Run the cell below with that file name. It extracts your code exactly the way
   the grader does and runs every test.
4. Upload the notebook to Moodle. Nothing else.


In [21]:
# Save the notebook before running this check: it grades the file on disk.
ir_sandbox.check_notebook("ma.chegodaeva@innopolis.university.ipynb")

Checking ma.chegodaeva@innopolis.university.ipynb

Task 1  Polite fetcher   20/20 points
  ok       2/2   fetch returns an HTML page with all fields
  ok       2/2   every request sends the User-Agent
  ok       2/2   binary files come back unchanged
  ok       3/3   disk cache: one entry per URL, reused by a new fetcher
  ok       2/2   error pages are reported and never cached
  ok       1/1   redirects are followed and cached under the requested URL
  ok       2/2   timeouts and connection errors give status None
  ok       2/2   delay between requests to the same host
  ok       4/4   download streams big files safely

Task 2  URL normalization and page parsing   20/20 points
  ok       6/6   normalize_url (6 groups of cases)
  ok       2/2   title, canonical and meta_robots
  ok       2/2   links: URLs in document order
  ok       1/1   links: anchor text
  ok       1/1   links: nofollow flags
  ok       2/2   images
  ok       3/3   paragraphs
  ok       2/2   visible text
  ok  

[{'task': 1,
  'title': 'Polite fetcher',
  'points': 20,
  'max': 20,
  'checks': [{'name': 't1_page',
    'title': 'fetch returns an HTML page with all fields',
    'points': 2,
    'max': 2,
    'message': '',
    'seconds': 0.0},
   {'name': 't1_user_agent',
    'title': 'every request sends the User-Agent',
    'points': 2,
    'max': 2,
    'message': '',
    'seconds': 0.0},
   {'name': 't1_binary',
    'title': 'binary files come back unchanged',
    'points': 2,
    'max': 2,
    'message': '',
    'seconds': 0.0},
   {'name': 't1_cache',
    'title': 'disk cache: one entry per URL, reused by a new fetcher',
    'points': 3,
    'max': 3,
    'message': '',
    'seconds': 0.1},
   {'name': 't1_errors',
    'title': 'error pages are reported and never cached',
    'points': 2,
    'max': 2,
    'message': '',
    'seconds': 0.0},
   {'name': 't1_redirect',
    'title': 'redirects are followed and cached under the requested URL',
    'points': 1,
    'max': 1,
    'message': '',